# Demo: Ingest a Materials Project record

This notebook demonstrates fetching a materials record using the `mp_api` client (via the adapter),
then ingesting it with the existing `IngestionPipeline` to preserve the raw payload and
produce a normalised `MaterialRecord`. Set the `MP_API_KEY` environment variable before running
a live fetch.

In [12]:
# Imports and setup

import os
from pathlib import Path

from dotenv import load_dotenv

from materials_pipeline.pipeline import IngestionPipeline
from materials_pipeline.providers.materials_project import MaterialsProjectProvider
from materials_pipeline.providers.materials_project_api import (
    MaterialsProjectAPIAdapter,
)

# Set the project root to the repository root (adjust if running elsewhere)
project_root = Path(".").cwd()

# Read Materials Project API key from .env file
load_dotenv()

# Read Materials Project API key from environment if available
mp_api_key = os.environ.get("MP_API_KEY")

print("MP_API_KEY present:", bool(mp_api_key))

MP_API_KEY present: True


In [9]:
# Create adapter and pipeline instances
adapter = MaterialsProjectAPIAdapter(api_key=mp_api_key)
pipeline = IngestionPipeline(project_root=project_root)
provider = MaterialsProjectProvider()

# Choose a Materials Project id to fetch (example id)
material_id = "mp-149"
print("Fetching", material_id)

Fetching mp-149


In [10]:
# Fetch the raw Materials Project record
raw = adapter.fetch_material(material_id)
print("Raw keys:", list(raw.keys()))
# For inspection, pretty-print the top-level raw fields (structure may be large)
print("formula_pretty =", raw.get("formula_pretty"))
print("chemsys =", raw.get("chemsys"))

Retrieving SummaryDoc documents: 100%|██████████| 1/1 [00:00<00:00, 17476.27it/s]

Raw keys: ['material_id', 'formula_pretty', 'chemsys', 'structure']
formula_pretty = Si
chemsys = Si


In [11]:
# Ingest the fetched record through the pipeline
result = pipeline.ingest(provider=provider, raw=raw)

# Show summary of the ingestion result
print("Ingestion status:", result.status)
if result.raw_path:
    print("Raw saved to:", result.raw_path)
if result.normalised_path:
    print("Normalised saved to:", result.normalised_path)

# If accepted, print a short representation of the normalised composition
if result.material is not None:
    material_dump = result.material.model_dump()
    print("Reduced formula:", material_dump["composition"]["reduced_formula"])
    print("Element amounts:", material_dump["composition"]["element_amounts"])

Ingestion status: accepted
Raw saved to: /var/home/connor/Python/$ Projects/materials-pipeline/notebooks/data/raw/materials_project/mp-149/20260901T153354981516Z.json
Normalised saved to: /var/home/connor/Python/$ Projects/materials-pipeline/notebooks/data/normalised/398c5b59-9904-4e9e-8a07-aa149b5e15f1.json
Reduced formula: Si
Element amounts: {'Si': 1.0}


## Notes
- Provide your Materials Project API key via the `MP_API_KEY` environment variable to fetch live data
  Example (Linux/macOS): `export MP_API_KEY=your_key_here`
  or by adding to to an `.env` file
  Example: `MP_API_KEY=your_key_here`
- This notebook writes raw payloads to `data/raw/<provider>/` and normalised records to `data/normalised/`.